In [ ]:
!pip install datasets

from datasets import load_dataset

dataset = load_dataset("moriire/symptoms2diseases")
df = dataset['train'].to_pandas()
print(df.shape)
print(df.head())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Symptom2Disease.csv:   0%|          | 0.00/230k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1200 [00:00<?, ? examples/s]

(1200, 3)
   Unnamed: 0      label                                               text
0           0  Psoriasis  I have been experiencing a skin rash on my arm...
1           1  Psoriasis  My skin has been peeling, especially on my kne...
2           2  Psoriasis  I have been experiencing joint pain in my fing...
3           3  Psoriasis  There is a silver like dusting on my skin, esp...
4           4  Psoriasis  My nails have small dents or pits in them, and...


In [ ]:
!pip install sentence-transformers faiss-cpu

from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Create embeddings for all symptom texts
print("Creating embeddings... this takes a minute")
texts = df['text'].tolist()
labels = df['label'].tolist()
embeddings = model.encode(texts, show_progress_bar=True)

print(f"Embeddings shape: {embeddings.shape}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 36.1 MB/s eta 0:00:00


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Creating embeddings... this takes a minute


Batches:   0%|          | 0/38 [00:00<?, ?it/s]

Embeddings shape: (1200, 384)


In [4]:
# Build FAISS index
dimension = embeddings.shape[1]  # 384
index = faiss.IndexFlatL2(dimension)
index.add(embeddings.astype(np.float32))

print(f"Total vectors in index: {index.ntotal}")

# Test retrieval
def retrieve_similar(query, top_k=3):
    query_embedding = model.encode([query])
    distances, indices = index.search(query_embedding.astype(np.float32), top_k)
    results = []
    for i, idx in enumerate(indices[0]):
        results.append({
            "text": texts[idx],
            "label": labels[idx],
            "distance": distances[0][i]
        })
    return results

# Test it
test_results = retrieve_similar("I have fever and sore throat")
for r in test_results:
    print(f"Disease: {r['label']}")
    print(f"Text: {r['text'][:100]}...")
    print()

Total vectors in index: 1200
Disease: Chicken pox
Text: I have a high fever and a mild headache. I'm tired most of the time and completely lost my appetite....

Disease: Bronchial Asthma
Text: doctor, i have been having high fever since past few days , saliva also became thick , dry cough , w...

Disease: Impetigo
Text: I am suffering from mild fever and headache. There are small sores near my nose and rashes on my nec...



In [6]:
!pip install anthropic


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 923.9/923.9 kB 25.6 MB/s eta 0:00:00


In [7]:

import anthropic

client = anthropic.Anthropic(api_key="youe-api-key")

def rag_symptom_checker(user_message, conversation_history):
    # Step 1: Retrieve relevant medical context
    retrieved = retrieve_similar(user_message, top_k=3)
    context = "\n".join([f"Disease: {r['label']}\nSymptoms: {r['text']}"
                         for r in retrieved])

    # Step 2: Add context to system prompt
    system_prompt = f"""You are a friendly medical assistant.
Use the following retrieved medical information to inform your response:

{context}

Ask follow-up questions, suggest possible conditions, recommend tests,
and always remind the user you are not a substitute for professional advice."""

    # Step 3: Call Claude with context
    response = client.messages.create(
        model="claude-opus-4-6",
        max_tokens=1024,
        system=system_prompt,
        messages=conversation_history
    )
    return response.content[0].text

In [8]:
import uuid
import json
from datetime import datetime

def chat_with_rag():
    conversation_history = []
    patient_name = input("Enter your name: ")
    patient_id = str(uuid.uuid4())[:6].upper()

    print(f"\nYour Patient ID is: {patient_id} — please save this!")
    print("Welcome to AI Health Assistant with RAG!")
    print("Type 'quit' to exit\n")
    print("-" * 50)

    while True:
        user_input = input("You: ")

        if user_input.lower() == 'quit':
            filename = f"conversation_{patient_name}_{patient_id}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
            with open(filename, 'w') as f:
                json.dump(conversation_history, f, indent=2)
            print(f"\nConversation saved to {filename}")
            print("Take care and stay healthy!")
            break

        conversation_history.append({
            "role": "user",
            "content": user_input
        })

        response = rag_symptom_checker(user_input, conversation_history)

        conversation_history.append({
            "role": "assistant",
            "content": response
        })

        print(f"\nDoctor AI: {response}\n")
        print("-" * 50)

chat_with_rag()

Enter your name: Disha Agarwal 

Your Patient ID is: 7BCE5F — please save this!
Welcome to AI Health Assistant with RAG!
Type 'quit' to exit

--------------------------------------------------
You: knee and calf pain, swelling in knee and ankle and difficulty in standing and walking 

Doctor AI: Thank you for sharing your symptoms with me. I'm sorry to hear you're experiencing this discomfort. Let me ask you some follow-up questions to better understand your situation.

## Follow-up Questions:

1. **Duration:** How long have you been experiencing these symptoms? Did they come on suddenly or gradually?
2. **Visible veins:** Can you see any swollen, twisted, or bulging blood vessels (veins) on your legs, particularly on your calves?
3. **Pain characteristics:** Is the pain a dull ache, throbbing, cramping, or sharp? Does it worsen after prolonged standing or sitting?
4. **Activity level:** What is your occupation? Do you stand or sit for long periods during the day?
5. **Medical history:

In [11]:
import os

def load_previous_conversation(patient_id):
    # Search all saved files for this patient ID
    files = [f for f in os.listdir() if patient_id in f]

    if files:
        # Load the most recent one
        latest_file = sorted(files)[-1]
        with open(latest_file, 'r') as f:
            conversation_history = json.load(f)
        print(f"Found previous conversation: {latest_file}")
        return conversation_history
    else:
        print("No previous conversation found. Starting fresh!")
        return []

In [14]:
name = input("Enter your name :")
rp = input("Are you a returning patient? (yes/no): ")
if rp.lower() == "yes":

    patient_id = input("Enter your patient ID: ")
    conversation_history = load_previous_conversation(patient_id)

else:
    conversation_history = []
    patient_id = str(uuid.uuid4())[:6].upper()

Enter your name :Disha agarwal 
Are you a returning patient? (yes/no): yes
Enter your patient ID: 7BCE5F
Found previous conversation: conversation_Disha Agarwal _7BCE5F_20260612_082551.json


In [19]:

print(f"\nWelcome {name}!")
print(f"Your Patient ID is: {patient_id} — please save this!")
print("Welcome to AI Health Assistant with RAG!")
print("Type 'quit' to exit\n")
print("-" * 50)

def chat_with_rag():
    name = input("Enter your name: ")
    rp = input("Are you a returning patient? (yes/no): ")

    if rp.lower() == "yes":
        patient_id = input("Enter your patient ID: ")
        conversation_history = load_previous_conversation(patient_id)
    else:
        conversation_history = []
        patient_id = str(uuid.uuid4())[:6].upper()

    print(f"\nWelcome {name}!")
    print(f"Your Patient ID is: {patient_id} — please save this!")
    print("Welcome to AI Health Assistant with RAG!")
    print("Type 'quit' to exit\n")
    print("-" * 50)

    while True:
        user_input = input("You: ")

        if user_input.lower() == 'quit':
            filename = f"conversation_{name}_{patient_id}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
            with open(filename, 'w') as f:
                json.dump(conversation_history, f, indent=2)
            print(f"\nConversation saved to {filename}")
            print("Take care and stay healthy!")
            break

        conversation_history.append({
            "role": "user",
            "content": user_input
        })

        response = rag_symptom_checker(user_input, conversation_history)

        conversation_history.append({
            "role": "assistant",
            "content": response
        })

        print(f"\nDoctor AI: {response}\n")
        print("-" * 50)

chat_with_rag()





Welcome Disha agarwal !
Your Patient ID is: 7BCE5F — please save this!
Welcome to AI Health Assistant with RAG!
Type 'quit' to exit

--------------------------------------------------
Enter your name: Siti
Are you a returning patient? (yes/no): no

Welcome Siti!
Your Patient ID is: 814482 — please save this!
Welcome to AI Health Assistant with RAG!
Type 'quit' to exit

--------------------------------------------------
You: stomach pain, difficulty in eating  and pain in right part of the body 

Doctor AI: Thank you for sharing your symptoms with me. I'm sorry to hear you're not feeling well. Let me ask you some follow-up questions to better understand your situation:

## Follow-up Questions:

1. **Stomach pain:** 
   - How long have you been experiencing this pain?
   - Is it a sharp, dull, cramping, or burning pain?
   - Is it constant or does it come and go?

2. **Difficulty eating:**
   - Do you feel nauseous or have vomiting?
   - Do you feel full quickly, or is it painful to eat

In [20]:
!pip install gradio

In [28]:
import gradio as gr

def gradio_chat(message, history, name, patient_id):
    if not name:
        return "Please enter your name first!"

    conversation_history = []
    for human, assistant in history:
        conversation_history.append({"role": "user", "content": human})
        conversation_history.append({"role": "assistant", "content": assistant})

    conversation_history.append({"role": "user", "content": message})
    response = rag_symptom_checker(message, conversation_history)
    return response

def save_conversation(history, name, patient_id):
    if not history:
        return "No conversation to save!"

    if not patient_id:
        patient_id = str(uuid.uuid4())[:6].upper()

    conversation_history = []
    for human, assistant in history:
        conversation_history.append({"role": "user", "content": human})
        conversation_history.append({"role": "assistant", "content": assistant})

    filename = f"conversation_{name}_{patient_id}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    with open(filename, 'w') as f:
        json.dump(conversation_history, f, indent=2)

    return f"Conversation saved! Your Patient ID is: {patient_id} — save this for future visits!"

with gr.Blocks(title="AI Health Assistant") as demo:
    gr.Markdown("# 🏥 AI Health Assistant")
    gr.Markdown("*Not a substitute for professional medical advice*")

    with gr.Row():
        name_input = gr.Textbox(label="Your Name", placeholder="Enter your name")
        patient_id_input = gr.Textbox(label="Patient ID", placeholder="Leave blank if new patient")

    chatbot = gr.ChatInterface(
        fn=gradio_chat,
        additional_inputs=[name_input, patient_id_input],
        title="",
        fill_height=False,
    )

    with gr.Row():
        save_btn = gr.Button("Save Conversation", variant="primary")
        save_output = gr.Textbox(label="", interactive=False)

    save_btn.click(
        fn=save_conversation,
        inputs=[chatbot.chatbot, name_input, patient_id_input],
        outputs=save_output
    )

demo.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5d5dbce6a53ae8b4ee.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
